# 🏠 Baseline Fuel Blend Property Prediction Model

**LightGBM Multi-Output Regression Approach**

This notebook implements a baseline model for predicting fuel blend properties using LightGBM with basic feature engineering.

## 📋 Project Overview
- **Dataset**: Fuel blend composition and component properties
- **Task**: Predict 10 blend properties from 69 input features
- **Input Features**: 5 component fractions + 50 component properties
- **Target**: 10 blend properties
- **Metric**: Mean Absolute Percentage Error (MAPE)
- **Model**: LightGBM with basic feature engineering

## Architecture Model Architecture
1. **Basic Feature Engineering**: Weighted blend features + statistical aggregates
2. **Model**: LightGBM MultiOutputRegressor
3. **Validation**: Train/Validation split (80/20)
4. **Hyperparameter Tuning**: Manual parameter selection

In [ ]:
# Install required packages
!pip install -q lightgbm scikit-learn matplotlib seaborn pandas numpy

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_percentage_error
import lightgbm as lgb
from sklearn.multioutput import MultiOutputRegressor
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("OK All libraries imported successfully!")

## Data Data Loading and Exploration

In [ ]:
# Load data from local data directory
train = pd.read_csv('data/train.csv')
test = pd.read_csv('data/test.csv')
sample_solution = pd.read_csv('data/sample_solution.csv')

print(f"Data Train shape: {train.shape}")
print(f"Data Test shape: {test.shape}")
print(f"Data Sample solution shape: {sample_solution.shape}")

# Display basic information
print("\nSearch Train data info:")
print(train.info())

print("\nChart Basic statistics:")
print(train.describe())

## Search Exploratory Data Analysis

In [ ]:
# Check blend composition sums
blend_cols = [f'Component{i}_fraction' for i in range(1, 6)]
print("Chemistry Blend composition validation:")
print(f"All blend fractions sum to ~1.0: {np.isclose(train[blend_cols].sum(axis=1), 1.0).all()}")
print(f"Blend fraction sums range: {train[blend_cols].sum(axis=1).min():.3f} - {train[blend_cols].sum(axis=1).max():.3f}")

# Target variables overview
target_cols = [f'BlendProperty{i}' for i in range(1, 11)]
print(f"\nTarget Target variables: {len(target_cols)}")
print("Target statistics:")
print(train[target_cols].describe())

In [ ]:
# Visualize feature correlations
plt.figure(figsize=(12, 10))
corr_matrix = train.corr()

# Show correlation between targets and some key features
target_corr = corr_matrix.loc[target_cols, blend_cols + [f'Component1_Property{i}' for i in range(1, 6)]]
sns.heatmap(target_corr, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Target vs Key Feature Correlations')
plt.tight_layout()
plt.show()

# Distribution of target variables
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.ravel()

for i, col in enumerate(target_cols):
    sns.histplot(train[col], ax=axes[i], kde=True)
    axes[i].set_title(f'{col} Distribution')
    axes[i].set_xlabel('')

plt.tight_layout()
plt.show()

## Chemistry Feature Engineering

In [ ]:
def create_weighted_blend_features(df):
    """Create blend-weighted features by multiplying fractions with properties"""
    for prop in range(1, 11):  # For each property type
        weighted_sum = 0
        for comp in range(1, 6):  # For each component
            frac_col = f"Component{comp}_fraction"
            prop_col = f"Component{comp}_Property{prop}"
            weighted_sum += df[frac_col] * df[prop_col]
        df[f"WeightedProperty{prop}"] = weighted_sum
    return df

def create_aggregate_features(df):
    """Add statistical aggregates of all component properties"""
    component_prop_cols = [f"Component{i}_Property{j}" for i in range(1, 6) for j in range(1, 11)]
    
    df["ComponentProp_mean"] = df[component_prop_cols].mean(axis=1)
    df["ComponentProp_std"] = df[component_prop_cols].std(axis=1)
    df["ComponentProp_min"] = df[component_prop_cols].min(axis=1)
    df["ComponentProp_max"] = df[component_prop_cols].max(axis=1)
    return df

# Apply feature engineering
print("Factory Applying feature engineering...")

# Create weighted features
train = create_weighted_blend_features(train)
test = create_weighted_blend_features(test)

# Create aggregate features
train = create_aggregate_features(train)
test = create_aggregate_features(test)

print(f"OK New train shape: {train.shape}")
print(f"OK New test shape: {test.shape}")

# Show new features
new_features = [col for col in train.columns if 'WeightedProperty' in col or 'ComponentProp_' in col]
print(f"Tool New features created: {len(new_features)}")
print(f"New feature names: {new_features}")

## AI Model Training

In [ ]:
# Prepare data for modeling
feature_cols = [col for col in train.columns if col not in target_cols]

X = train[feature_cols]
y = train[target_cols]

# Train/validation split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Data Training set: {X_train.shape}")
print(f"Data Validation set: {X_val.shape}")
print(f"Search Number of features: {len(feature_cols)}")
print(f"Target Number of targets: {len(target_cols)}")

# Show feature importance (correlation with targets)
feature_target_corr = X_train.corrwith(y_train.mean(axis=1)).abs().sort_values(ascending=False)
print("\nSearch Top 10 most correlated features with average target:")
print(feature_target_corr.head(10))

In [ ]:
# Initialize LightGBM model with tuned parameters
lgb_model = lgb.LGBMRegressor(
    objective='regression',
    learning_rate=0.05,
    n_estimators=300,
    max_depth=7,
    num_leaves=64,
    subsample=0.8,
    colsample_bytree=0.8,
    n_jobs=-1,
    random_state=42,
    verbose=-1
)

# Create multi-output regressor
multi_model = MultiOutputRegressor(lgb_model)

print("Launch Training LightGBM multi-output model...")
multi_model.fit(X_train, y_train)

# Make validation predictions
val_preds = multi_model.predict(X_val)

# Calculate validation MAPE
val_mape = mean_absolute_percentage_error(y_val, val_preds) * 100
print(f"OK Validation MAPE: {val_mape:.2f}%")

# Individual target MAPEs
print("\nData MAPE for each target variable:")
for i, target in enumerate(target_cols):
    target_mape = mean_absolute_percentage_error(y_val[target], val_preds[:, i]) * 100
    print(f"{target}: {target_mape:.2f}%")

## Target Predictions and Submission

In [ ]:
# Prepare test data (remove ID if present)
test_features = test.copy()
if 'ID' in test_features.columns:
    test_features = test_features.drop(columns=['ID'])

# Ensure feature alignment
test_features = test_features[feature_cols]

print(f"Lab Test features shape: {test_features.shape}")
print(f"Search Feature alignment check: {list(test_features.columns) == feature_cols}")

# Make predictions
test_preds = multi_model.predict(test_features)

# Create submission dataframe
submission = pd.DataFrame(test_preds, columns=target_cols)

# Add ID column if present in sample solution
if 'ID' in sample_solution.columns:
    submission.insert(0, 'ID', sample_solution['ID'])

# Round predictions to reasonable precision
submission[target_cols] = submission[target_cols].round(6)

# Save submission
submission.to_csv('baseline_submission.csv', index=False)

print("OK Submission saved as 'baseline_submission.csv'")
print(f"Data Submission shape: {submission.shape}")

# Preview submission
print("\nSearch Submission preview:")
print(submission.head())

## Data Model Performance Analysis

In [ ]:
# Calculate detailed performance metrics
print("Chart Baseline Model Performance Analysis")
print("=" * 50)

# Overall MAPE
overall_mape = mean_absolute_percentage_error(y_val, val_preds) * 100
print(f"Overall MAPE: {overall_mape:.2f}%")

# Individual target performance
target_mapes = []
for i, target in enumerate(target_cols):
    mape = mean_absolute_percentage_error(y_val[target], val_preds[:, i]) * 100
    target_mapes.append(mape)
    print(f"{target}: {mape:.2f}%")

print(f"\nData Performance Summary:")
print(f"Best performing target: {target_cols[np.argmin(target_mapes)]} ({min(target_mapes):.2f}%)")
print(f"Worst performing target: {target_cols[np.argmax(target_mapes)]} ({max(target_mapes):.2f}%)")
print(f"Average target MAPE: {np.mean(target_mapes):.2f}%")
print(f"MAPE standard deviation: {np.std(target_mapes):.2f}%")

# Feature importance analysis (using first estimator)
feature_importance = multi_model.estimators_[0].feature_importances_
importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': feature_importance
}).sort_values('importance', ascending=False)

print(f"\nSearch Top 10 most important features:")
for _, row in importance_df.head(10).iterrows():
    print(f"{row['feature']}: {row['importance']}")

# Visualize feature importance
plt.figure(figsize=(12, 8))
sns.barplot(data=importance_df.head(20), x='importance', y='feature')
plt.title('Top 20 Feature Importance (LightGBM)')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

## Target Key Insights and Next Steps

In [ ]:
print("Complete Baseline Fuel Blend Prediction Model - Complete!")
print("=" * 60)
print("\nData Model Summary:")
print(f"• Validation MAPE: {overall_mape:.2f}%")
print(f"• Number of features: {len(feature_cols)}")
print(f"• Number of targets: {len(target_cols)}")
print(f"• Training samples: {len(X_train)}")
print(f"• Validation samples: {len(X_val)}")

print("\nTool Features Used:")
print("• Original features: blend fractions + component properties")
print("• Engineered features: weighted blend properties + statistical aggregates")

print("\nChart Potential Improvements:")
print("• Physics-based features (linear blending rules)")
print("• Hyperparameter optimization (Optuna)")
print("• Ensemble methods (multiple models)")
print("• Advanced feature engineering (non-linear effects)")
print("• Cross-validation for more robust evaluation")

print("\nInsight Key Findings:")
print("• Some targets are much harder to predict than others")
print("• Blend composition features are highly important")
print("• Component properties show varying predictive power")
print("• Weighted features capture important blend-property relationships")